In [1]:
import openai

from typing import List, Iterator
import pandas as pd
import numpy as np
import os
import pickle
from ast import literal_eval

# Redis client library for Python
import redis
from redis.commands.search.indexDefinition import (
    IndexDefinition,
    IndexType
)
from redis.commands.search.query import Query
from redis.commands.search.field import (
    TextField,
    VectorField
)

# I've set this to our new embeddings model, this can be changed to the embedding model of your choice
EMBEDDING_MODEL = "text-embedding-3-small"

# Ignore unclosed SSL socket warnings - optional in case you get these errors
import warnings

warnings.filterwarnings(action="ignore", message="unclosed", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning) 

#### Load Data

In [2]:
transcript_df = pd.read_csv('data/embeddings.csv')

In [3]:
transcript_df.head()

,vector_id,id,video_id,chunk_id,content_vector,title_vector
0,0,0,q-wRvsiGYIs,_videoid:q-wRvsiGYIs_chunk:0,"[0.006686230190098286, -0.03623957931995392, -...","[0.027011631056666374, 0.019285723567008972, -..."
1,1,0,q-wRvsiGYIs,_videoid:q-wRvsiGYIs_chunk:1,"[0.02565005235373974, -0.008785320445895195, -...","[0.027011631056666374, 0.019285723567008972, -..."
2,2,0,q-wRvsiGYIs,_videoid:q-wRvsiGYIs_chunk:2,"[0.03873475268483162, -0.03571273759007454, -0...","[0.027011631056666374, 0.019285723567008972, -..."
3,3,0,q-wRvsiGYIs,_videoid:q-wRvsiGYIs_chunk:3,"[0.019842883571982384, -0.019223583862185478, ...","[0.027011631056666374, 0.019285723567008972, -..."
4,4,0,q-wRvsiGYIs,_videoid:q-wRvsiGYIs_chunk:4,"[0.005661938339471817, -0.05714607611298561, -...","[0.027011631056666374, 0.019285723567008972, -..."


In [4]:
# Load title dictionary from pickle file
with open('data/title_dict.pkl', 'rb') as f:
    title_dict = pickle.load(f)

title_df = pd.DataFrame(title_dict.items(), columns=['video_id', 'title'])
title_df.head()

,video_id,title
0,q-wRvsiGYIs,"AMA #19: Collagen vs. Whey Protein, Creatine, ..."
1,ssmwxKPFMFU,Protocols to Improve Vision & Eyesight | Huber...
2,J7yn4tJEmJU,Tools for Overcoming Substance & Behavioral Ad...
3,7MEhDlw1e9k,How to Build Endurance | Huberman Lab Essentials
4,UyneMnERmnI,How to Improve Your Vitality & Heal From Disea...


In [5]:
# Load title dictionary from pickle file
with open('data/chunk_dict.pkl', 'rb') as f:
    chunk_dict = pickle.load(f)

chunk_df = pd.DataFrame(chunk_dict.items(), columns=['chunk_id', 'text'])
chunk_df.head()

,chunk_id,text
0,_videoid:q-wRvsiGYIs_chunk:0,ANDREW HUBERMAN: Welcome to the Huberman Lab p...
1,_videoid:q-wRvsiGYIs_chunk:1,"is, as well as its quality or protein score re..."
2,_videoid:q-wRvsiGYIs_chunk:2,in a way that either mimics or can replace the...
3,_videoid:q-wRvsiGYIs_chunk:3,"like rice and oatmeal and things like that, if..."
4,_videoid:q-wRvsiGYIs_chunk:4,"get the idea, if you are experiencing troublin..."


In [6]:
# Join transcript_df with title_df on video_id to add titles
transcript_df = transcript_df.merge(title_df, on='video_id', how='left')
transcript_df = transcript_df.merge(chunk_df, on='chunk_id', how='left')
transcript_df.head()

,vector_id,id,video_id,chunk_id,content_vector,title_vector,title,text
0,0,0,q-wRvsiGYIs,_videoid:q-wRvsiGYIs_chunk:0,"[0.006686230190098286, -0.03623957931995392, -...","[0.027011631056666374, 0.019285723567008972, -...","AMA #19: Collagen vs. Whey Protein, Creatine, ...",ANDREW HUBERMAN: Welcome to the Huberman Lab p...
1,1,0,q-wRvsiGYIs,_videoid:q-wRvsiGYIs_chunk:1,"[0.02565005235373974, -0.008785320445895195, -...","[0.027011631056666374, 0.019285723567008972, -...","AMA #19: Collagen vs. Whey Protein, Creatine, ...","is, as well as its quality or protein score re..."
2,2,0,q-wRvsiGYIs,_videoid:q-wRvsiGYIs_chunk:2,"[0.03873475268483162, -0.03571273759007454, -0...","[0.027011631056666374, 0.019285723567008972, -...","AMA #19: Collagen vs. Whey Protein, Creatine, ...",in a way that either mimics or can replace the...
3,3,0,q-wRvsiGYIs,_videoid:q-wRvsiGYIs_chunk:3,"[0.019842883571982384, -0.019223583862185478, ...","[0.027011631056666374, 0.019285723567008972, -...","AMA #19: Collagen vs. Whey Protein, Creatine, ...","like rice and oatmeal and things like that, if..."
4,4,0,q-wRvsiGYIs,_videoid:q-wRvsiGYIs_chunk:4,"[0.005661938339471817, -0.05714607611298561, -...","[0.027011631056666374, 0.019285723567008972, -...","AMA #19: Collagen vs. Whey Protein, Creatine, ...","get the idea, if you are experiencing troublin..."


In [7]:
# Read vectors from strings back into a list
transcript_df['title_vector'] = transcript_df.title_vector.apply(literal_eval)
transcript_df['content_vector'] = transcript_df.content_vector.apply(literal_eval)

# Set vector_id to be a string
transcript_df['vector_id'] = transcript_df['vector_id'].apply(str)

In [8]:
transcript_df

,vector_id,id,video_id,chunk_id,content_vector,title_vector,title,text
0,0,0,q-wRvsiGYIs,_videoid:q-wRvsiGYIs_chunk:0,"[0.006686230190098286, -0.03623957931995392, -...","[0.027011631056666374, 0.019285723567008972, -...","AMA #19: Collagen vs. Whey Protein, Creatine, ...",ANDREW HUBERMAN: Welcome to the Huberman Lab p...
1,1,0,q-wRvsiGYIs,_videoid:q-wRvsiGYIs_chunk:1,"[0.02565005235373974, -0.008785320445895195, -...","[0.027011631056666374, 0.019285723567008972, -...","AMA #19: Collagen vs. Whey Protein, Creatine, ...","is, as well as its quality or protein score re..."
2,2,0,q-wRvsiGYIs,_videoid:q-wRvsiGYIs_chunk:2,"[0.03873475268483162, -0.03571273759007454, -0...","[0.027011631056666374, 0.019285723567008972, -...","AMA #19: Collagen vs. Whey Protein, Creatine, ...",in a way that either mimics or can replace the...
3,3,0,q-wRvsiGYIs,_videoid:q-wRvsiGYIs_chunk:3,"[0.019842883571982384, -0.019223583862185478, ...","[0.027011631056666374, 0.019285723567008972, -...","AMA #19: Collagen vs. Whey Protein, Creatine, ...","like rice and oatmeal and things like that, if..."
4,4,0,q-wRvsiGYIs,_videoid:q-wRvsiGYIs_chunk:4,"[0.005661938339471817, -0.05714607611298561, -...","[0.027011631056666374, 0.019285723567008972, -...","AMA #19: Collagen vs. Whey Protein, Creatine, ...","get the idea, if you are experiencing troublin..."
...,...,...,...,...,...,...,...,...
673,673,29,qUz93CyNIz0,_videoid:qUz93CyNIz0_chunk:2,"[0.018683400005102158, 0.006987184286117554, -...","[-0.037588391453027725, 0.016762753948569298, ...",Tools for Managing Stress & Anxiety | Huberman...,stress is that it works in real time this does...
674,674,29,qUz93CyNIz0,_videoid:qUz93CyNIz0_chunk:3,"[0.0009851709473878145, 0.012981480918824673, ...","[-0.037588391453027725, 0.016762753948569298, ...",Tools for Managing Stress & Anxiety | Huberman...,and long-term stress but I want to say short-t...
675,675,29,qUz93CyNIz0,_videoid:qUz93CyNIz0_chunk:4,"[0.018381869420409203, 0.01727227307856083, 0....","[-0.037588391453027725, 0.016762753948569298, ...",Tools for Managing Stress & Anxiety | Huberman...,good sleep what good sleep means to you please...
676,676,29,qUz93CyNIz0,_videoid:qUz93CyNIz0_chunk:5,"[-0.0004088794521521777, -0.024336257949471474...","[-0.037588391453027725, 0.016762753948569298, ...",Tools for Managing Stress & Anxiety | Huberman...,is everyone knows getting regular exercise get...


In [9]:
transcript_df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 678 entries, 0 to 677
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   vector_id       678 non-null    object
 1   id              678 non-null    int64 
 2   video_id        678 non-null    object
 3   chunk_id        678 non-null    object
 4   content_vector  678 non-null    object
 5   title_vector    678 non-null    object
 6   title           678 non-null    object
 7   text            678 non-null    object
dtypes: int64(1), object(7)
memory usage: 42.5+ KB


### Redis

#### Setup

In [10]:
REDIS_HOST =  "localhost"
REDIS_PORT = 6379
REDIS_PASSWORD = "" # default for passwordless Redis

# Connect to Redis
redis_client = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    password=REDIS_PASSWORD
)
redis_client.ping()

True

#### Creating a Search Index

The below cells will show how to specify and create a search index in Redis. We will

1. Set some constants for defining our index like the distance metric and the index name
2. Define the index schema with RediSearch fields
3. Create the index


In [11]:
# Constants
VECTOR_DIM = len(transcript_df['title_vector'][0]) # length of the vectors
VECTOR_NUMBER = len(transcript_df)                 # initial number of vectors
INDEX_NAME = "embeddings-index"                    # name of the search index
PREFIX = "doc"                                     # prefix for the document keys
DISTANCE_METRIC = "COSINE"                         # distance metric for the vectors (ex. COSINE, IP, L2)

In [12]:
# Define RediSearch fields for each of the columns in the dataset
title = TextField(name="title")
text = TextField(name="text")
video_id = TextField(name="video_id")
chunk_id = TextField(name="chunk_id")
# url = TextField(name="url")

title_embedding = VectorField("title_vector",
    "FLAT", {
        "TYPE": "FLOAT32",
        "DIM": VECTOR_DIM,
        "DISTANCE_METRIC": DISTANCE_METRIC,
        "INITIAL_CAP": VECTOR_NUMBER,
    }
)
text_embedding = VectorField("content_vector",
    "FLAT", {
        "TYPE": "FLOAT32",
        "DIM": VECTOR_DIM,
        "DISTANCE_METRIC": DISTANCE_METRIC,
        "INITIAL_CAP": VECTOR_NUMBER,
    }
)
fields = [title, text, video_id, chunk_id, title_embedding, text_embedding]

In [ ]:
# Check if index exists
try:
    redis_client.ft(INDEX_NAME).info()
    print("Index already exists")
except:
    # Create RediSearch Index
    redis_client.ft(INDEX_NAME).create_index(
        fields = fields,
        definition = IndexDefinition(prefix=[PREFIX], index_type=IndexType.HASH)
    )

#### Load Documents into Index

In [14]:
def index_documents(client: redis.Redis, prefix: str, documents: pd.DataFrame):
    records = documents.to_dict("records")
    for doc in records:
        key = f"{prefix}:{str(doc['id'])}"

        # create byte vectors for title and content
        title_embedding = np.array(doc["title_vector"], dtype=np.float32).tobytes()
        content_embedding = np.array(doc["content_vector"], dtype=np.float32).tobytes()

        # replace list of floats with byte vectors
        doc["title_vector"] = title_embedding
        doc["content_vector"] = content_embedding

        client.hset(key, mapping = doc)

In [15]:
index_documents(redis_client, PREFIX, transcript_df)
print(f"Loaded {redis_client.info()['db0']['keys']} documents in Redis search index with name: {INDEX_NAME}")

Loaded 30 documents in Redis search index with name: embeddings-index


#### Running Search Queries

In [49]:
def search_redis(
    openai_client: openai.OpenAI,
    redis_client: redis.Redis,
    user_query: str,
    index_name: str = "embeddings-index",
    vector_field: str = "title_vector",
    return_fields: list = ["title", "text", "chunk_id", "vector_score"],
    hybrid_fields = "*",
    k: int = 20,
) -> List[dict]:

    # Creates embedding vector from user query
    embedded_query = openai_client.embeddings.create(input=user_query,
                                            model=EMBEDDING_MODEL,
                                            ).data[0].embedding

    # Prepare the Query
    base_query = f'{hybrid_fields}=>[KNN {k} @{vector_field} $vector AS vector_score]'
    query = (
        Query(base_query)
         .return_fields(*return_fields)
         .sort_by("vector_score")
         .paging(0, k)
         .dialect(2)
    )
    params_dict = {"vector": np.array(embedded_query).astype(dtype=np.float32).tobytes()}

    # perform vector search
    results = redis_client.ft(index_name).search(query, params_dict)
    for i, article in enumerate(results.docs):
        score = 1 - float(article.vector_score)
        print(f"{i}. {article.title}\n\t (Score: {round(score ,3) })")
    return results.docs

In [50]:
# For using OpenAI to generate query embedding
from dotenv import load_dotenv
load_dotenv()
open_api_key = os.getenv("OPENAI_API_KEY")
openai_client = openai.OpenAI(api_key=open_api_key)

In [57]:
results = search_redis(
    openai_client, 
    redis_client, 
    'dopamine in the brain', 
    k=10
)

0. How to Increase Motivation & Drive | Huberman Lab Essentials
	 (Score: 0.27)
1. How Hormones Shape Sexual Development | Huberman Lab Essentials
	 (Score: 0.266)
2. Improve Focus with Behavioral Tools & Medication for ADHD | Dr. John Kruse
	 (Score: 0.252)
3. How Foods & Nutrients Control Our Moods | Huberman Lab Essentials
	 (Score: 0.249)
4. Using Your Mind to Control Your Physical Health & Longevity | Dr. Ellen Langer
	 (Score: 0.224)
5. The Science of Emotions & Relationships | Huberman Lab Essentials
	 (Score: 0.218)
6. Boost Your Energy & Immune System with Cortisol & Adrenaline | Huberman Lab Essentials
	 (Score: 0.2)
7. How to Control Hunger, Eating & Satiety | Huberman Lab Essentials
	 (Score: 0.198)
8. How to Improve Your Teeth & Oral Microbiome for Brain & Body Health | Dr. Staci Whitman
	 (Score: 0.188)
9. How to Control Your Metabolism by Thyroid & Growth Hormone | Huberman Lab Essentials
	 (Score: 0.185)


In [58]:
results[:2]

[Document {'id': 'doc:25', 'payload': None, 'vector_score': '0.729629397392', 'title': 'How to Increase Motivation & Drive | Huberman Lab Essentials', 'text': "that every once in a while gives you a win to keep you playing this is the the probability of winning on the craps table or the roulette table or at Blackjack just often enough that you're willing to buy tickets head out there play again go downstairs again from your room even though you swear you were done for the night intermittent reinforcement is the most powerful form of dopamine reward schedule to keep you doing something so we can export that we can use it for good if there's something that you're pursuing in life whether or not it's an academic goal or a financial goal or relationship goal one of the things that you can do to ensure that you will remain on the path to that goal for a very long time and that you will continue to exceed your previous performance as well as continue to enjoy the dopamine Rel that occurs whe

In [53]:
results = search_redis(
    openai_client, 
    redis_client, 
    'Sleep and Exercise', 
    vector_field='content_vector', 
    k=10
)

0. Lose Fat With Science-Based Tools | Huberman Lab Essentials
	 (Score: 0.402)
1. How to Optimize Testosterone & Estrogen | Huberman Lab Essentials
	 (Score: 0.385)
2. How to Learn Skills Faster | Huberman Lab Essentials
	 (Score: 0.367)
3. Supercharge Exercise Performance & Recovery with Cooling | Huberman Lab Essentials
	 (Score: 0.357)
4. How to Build Endurance | Huberman Lab Essentials
	 (Score: 0.356)
5. Build Muscle Size, Increase Strength & Improve Recovery | Huberman Lab Essentials
	 (Score: 0.307)
6. Protocols to Improve Vision & Eyesight | Huberman Lab Essentials
	 (Score: 0.302)
7. How to Improve Your Teeth & Oral Microbiome for Brain & Body Health | Dr. Staci Whitman
	 (Score: 0.299)
8. How to Build Strength, Endurance & Flexibility at Any Age | Pavel Tsatsouline
	 (Score: 0.298)
9. Tools for Managing Stress & Anxiety | Huberman Lab Essentials
	 (Score: 0.298)


#### Hybrid Queries with Redis

In [54]:
def create_hybrid_field(field_name: str, value: str) -> str:
    return f'@{field_name}:"{value}"'

In [55]:
# search the content vector for articles about famous battles in Scottish history and only include results with Scottish in the title
results = search_redis(openai_client,
                       redis_client,
                       "Sleep and Exercise",
                       vector_field="title_vector",
                       k=5,
                       hybrid_fields=create_hybrid_field("title", "Huberman")
                       )

0. Supercharge Exercise Performance & Recovery with Cooling | Huberman Lab Essentials
	 (Score: 0.363)
1. Boost Your Energy & Immune System with Cortisol & Adrenaline | Huberman Lab Essentials
	 (Score: 0.348)
2. How to Build Endurance | Huberman Lab Essentials
	 (Score: 0.348)
3. Build Muscle Size, Increase Strength & Improve Recovery | Huberman Lab Essentials
	 (Score: 0.33)
4. How to Control Hunger, Eating & Satiety | Huberman Lab Essentials
	 (Score: 0.323)


In [59]:
# run a hybrid query for articles about Art in the title vector and only include results with the phrase "Leonardo da Vinci" in the text
results = search_redis(openai_client,
                       redis_client,
                       "Sleep and Exercise",
                       vector_field="title_vector",
                       k=5,
                       hybrid_fields=create_hybrid_field("text", "Huberman")
                       )
# find specific mention of Leonardo da Vinci in the text that our full-text-search query returned
mention = [sentence for sentence in results[0].text.split("\n") if "Huberman" in sentence][0]
mention

0. How to Enhance Your Immune System | Dr. Roger Seheult
	 (Score: 0.298)
1. Improve Focus with Behavioral Tools & Medication for ADHD | Dr. John Kruse
	 (Score: 0.294)
2. How to Build Strength, Endurance & Flexibility at Any Age | Pavel Tsatsouline
	 (Score: 0.286)
3. Using Your Mind to Control Your Physical Health & Longevity | Dr. Ellen Langer
	 (Score: 0.274)
4. AMA #19: Collagen vs. Whey Protein, Creatine, Smelling Salts, Stimulants & More
	 (Score: 0.27)
